In [ ]:
# CELL 1: IMPORT LIBRARIES

import os
import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [ ]:
# CELL 2: DEFINE PATHS

BASE_PATH = "dataset/processed_keypoints"

CLASSES = ["danger", "help"]

data = []
labels = []

In [ ]:
# CELL 3: LOAD .npy KEYPOINT FILES

for label_idx, cls in enumerate(CLASSES):
    class_path = os.path.join(BASE_PATH, cls)
    
    for file in os.listdir(class_path):
        if file.endswith(".npy"):
            filepath = os.path.join(class_path, file)
            keypoints = np.load(filepath)
            
            data.append(keypoints)
            labels.append(label_idx)

print("Total samples:", len(data))

In [ ]:
import os
print(os.getcwd())

In [ ]:
import os

os.chdir("/Users/venubandi/Desktop/Gesture_Project")
print(os.getcwd())

In [ ]:
# CELL 4: CHECK SHAPES

print("Example shape:", data[0].shape)
print("First label:", labels[0])

In [ ]:
# CELL 5: CONVERT DATA TO ARRAYS

X = np.array(data)
y = np.array(labels)

print("X shape:", X.shape)
print("y shape:", y.shape)

In [ ]:
# CELL 6: TRAIN/TEST SPLIT

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

In [ ]:
# CELL 7: CONVERT NUMPY ARRAYS TO PYTORCH TENSORS

import torch

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

print("X_train_tensor:", X_train_tensor.shape)
print("y_train_tensor:", y_train_tensor.shape)

In [ ]:
# CELL 8: DEFINE LSTM MODEL

import torch.nn as nn

class GestureLSTM(nn.Module):
    def __init__(self, input_size=63, hidden_size=64, num_classes=2):
        super(GestureLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_output = lstm_out[:, -1, :]
        output = self.fc(last_output)
        return output

model = GestureLSTM()

print(model)

In [ ]:
# CELL 9: LOSS FUNCTION AND OPTIMIZER

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Model ready for training")

In [ ]:
# CELL 10: TRAIN THE LSTM MODEL

EPOCHS = 30

train_losses = []

for epoch in range(EPOCHS):
    model.train()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())

    print(f"Epoch [{epoch+1}/{EPOCHS}] - Loss: {loss.item():.4f}")

In [ ]:
# CELL 11: EVALUATE MODEL ON TEST SET

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model.eval()

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, test_predictions = torch.max(test_outputs, 1)

accuracy = accuracy_score(y_test_tensor.numpy(), test_predictions.numpy())

print("Test Accuracy:", round(accuracy * 100, 2), "%")
print("\nClassification Report:")
print(classification_report(
    y_test_tensor.numpy(),
    test_predictions.numpy(),
    target_names=["DANGER", "HELP"]
))

print("Confusion Matrix:")
print(confusion_matrix(y_test_tensor.numpy(), test_predictions.numpy()))

In [ ]:
# CELL 12: SAVE TRAINED MODEL WEIGHTS

MODEL_PATH = "gesture_lstm_help_danger.pth"

torch.save(model.state_dict(), MODEL_PATH)

print("Model weights saved successfully")
print("Saved file:", MODEL_PATH)

In [ ]:
# CELL 13: LOAD MODEL WEIGHTS FOR INFERENCE

loaded_model = GestureLSTM()
loaded_model.load_state_dict(torch.load(MODEL_PATH))
loaded_model.eval()

print("Weights loaded successfully")
print("Model ready for inference")

In [ ]:
# CELL 14: SINGLE SAMPLE INFERENCE

import torch.nn.functional as F

label_map = {0: "DANGER", 1: "HELP"}

sample_index = 0
sample = X_test_tensor[sample_index].unsqueeze(0)
actual_label = y_test_tensor[sample_index].item()

with torch.no_grad():
    output = loaded_model(sample)
    probabilities = F.softmax(output, dim=1)
    confidence, predicted_class = torch.max(probabilities, 1)

print("Actual Label:", label_map[actual_label])
print("Predicted Label:", label_map[predicted_class.item()])
print("Confidence:", round(confidence.item() * 100, 2), "%")

In [ ]:
# CELL 15: MEASURE INFERENCE SPEED

import time

num_runs = 100
sample = X_test_tensor[0].unsqueeze(0)

start_time = time.time()

with torch.no_grad():
    for _ in range(num_runs):
        _ = loaded_model(sample)

end_time = time.time()

avg_time = (end_time - start_time) / num_runs
fps = 1 / avg_time

print("Average inference time:", round(avg_time, 6), "seconds")
print("Estimated FPS:", round(fps, 2))

In [ ]:
# CELL 16: FIND SAMPLE RAW VIDEO AND MATCHING KEYPOINT FILE

import os

CLASS_NAME = "help"   # change to "danger" if you want

raw_dir = f"dataset/raw_videos/{CLASS_NAME}"
kp_dir = f"dataset/processed_keypoints/{CLASS_NAME}"

raw_videos = sorted([f for f in os.listdir(raw_dir) if f.lower().endswith(".mov")])
keypoints = sorted([f for f in os.listdir(kp_dir) if f.lower().endswith(".npy")])

print("Raw videos:", raw_videos[:5])
print("Keypoint files:", keypoints[:5])

sample_video = os.path.join(raw_dir, raw_videos[0])
sample_keypoint = os.path.join(kp_dir, keypoints[0])

print("Selected video:", sample_video)
print("Selected keypoint:", sample_keypoint)

In [ ]:
# CELL 17: DISPLAY RAW VIDEO

from IPython.display import Video, display

display(Video(sample_video, embed=True, width=400))

In [ ]:
# CELL 18: LOAD KEYPOINT SEQUENCE

kp = np.load(sample_keypoint)

print("Keypoint shape:", kp.shape)
print("First frame sample values:", kp[0][:10])

In [ ]:
# CELL 19: VISUALIZE KEYPOINTS FRAME BY FRAME

import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

# MediaPipe hand connections
HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12),
    (9,13),(13,14),(14,15),(15,16),
    (13,17),(17,18),(18,19),(19,20),
    (0,17)
]

# reshape from (30, 63) to (30, 21, 3)
kp_reshaped = kp.reshape(kp.shape[0], 21, 3)

for frame_idx in range(kp_reshaped.shape[0]):
    clear_output(wait=True)

    frame = kp_reshaped[frame_idx]
    x = frame[:, 0]
    y = frame[:, 1]

    plt.figure(figsize=(5, 5))
    plt.scatter(x, y)

    for start, end in HAND_CONNECTIONS:
        plt.plot([x[start], x[end]], [y[start], y[end]])

    plt.gca().invert_yaxis()
    plt.title(f"{CLASS_NAME.upper()} - Extracted Keypoints - Frame {frame_idx+1}")
    plt.xlabel("X")
    plt.ylabel("Y")
    plt.show()

    time.sleep(0.08)

In [ ]:
# CELL 19: CREATE VIDEO WITH KEYPOINTS DRAWN ON PERSON

import cv2
import numpy as np
import os

CLASS_NAME = "help"   # change to "danger" if needed

raw_dir = f"dataset/raw_videos/{CLASS_NAME}"
kp_dir = f"dataset/processed_keypoints/{CLASS_NAME}"

raw_videos = sorted([f for f in os.listdir(raw_dir) if f.lower().endswith(".mov")])
keypoints = sorted([f for f in os.listdir(kp_dir) if f.lower().endswith(".npy")])

sample_video = os.path.join(raw_dir, raw_videos[0])
sample_keypoint = os.path.join(kp_dir, keypoints[0])

kp = np.load(sample_keypoint).reshape(-1, 21, 3)

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(9,11),(11,12),
    (9,13),(13,14),(14,15),(15,16),
    (13,17),(17,18),(18,19),(19,20),
    (0,17)
]

cap = cv2.VideoCapture(sample_video)

fps = cap.get(cv2.CAP_PROP_FPS)
if fps == 0:
    fps = 30

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

output_path = f"{CLASS_NAME}_keypoint_overlay.mp4"

fourcc = cv2.VideoWriter_fourcc(*"mp4v")
out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

frame_idx = 0

while cap.isOpened() and frame_idx < len(kp):
    ret, frame = cap.read()
    if not ret:
        break

    landmarks = kp[frame_idx]

    for start, end in HAND_CONNECTIONS:
        x1 = int(landmarks[start][0] * width)
        y1 = int(landmarks[start][1] * height)
        x2 = int(landmarks[end][0] * width)
        y2 = int(landmarks[end][1] * height)

        cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    for point in landmarks:
        x = int(point[0] * width)
        y = int(point[1] * height)
        cv2.circle(frame, (x, y), 4, (0, 0, 255), -1)

    cv2.putText(
        frame,
        f"Class: {CLASS_NAME.upper()} | Frame: {frame_idx+1}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2
    )

    out.write(frame)
    frame_idx += 1

cap.release()
out.release()

print("Overlay video saved:", output_path)

In [ ]:
# CELL 20: DISPLAY KEYPOINT OVERLAY VIDEO

from IPython.display import Video, display

display(Video(output_path, embed=True, width=500))

In [ ]:
sample_video = os.path.join(raw_dir, raw_videos[0])
sample_keypoint = os.path.join(kp_dir, keypoints[0])

In [ ]:
print("Video used:", raw_videos[0])
print("Keypoint file used:", keypoints[0])

In [ ]:
# CELL: GENERATE OVERLAY VIDEOS FOR HELP + DANGER

import cv2
import numpy as np
import os

CLASSES = ["help", "danger"]

HAND_CONNECTIONS = [
    (0,1),(1,2),(2,3),(3,4),
    (0,5),(5,6),(6,7),(7,8),
    (5,9),(9,10),(10,11),(11,12),
    (9,13),(13,14),(14,15),(15,16),
    (13,17),(17,18),(18,19),(19,20),
    (0,17)
]

for CLASS_NAME in CLASSES:
    print(f"\nProcessing {CLASS_NAME.upper()}...")

    raw_dir = f"dataset/raw_videos/{CLASS_NAME}"
    kp_dir = f"dataset/processed_keypoints/{CLASS_NAME}"

    raw_videos = sorted([f for f in os.listdir(raw_dir) if f.lower().endswith(".mov")])

    # pick FIRST video (you can change index later)
    file_name = raw_videos[0]

    video_base = os.path.splitext(file_name)[0]
    kp_file = video_base + ".npy"

    sample_video = os.path.join(raw_dir, file_name)
    sample_keypoint = os.path.join(kp_dir, kp_file)

    print("Video:", file_name)
    print("Keypoints:", kp_file)

    kp = np.load(sample_keypoint).reshape(-1, 21, 3)

    cap = cv2.VideoCapture(sample_video)

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0:
        fps = 30

    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    output_path = f"{CLASS_NAME}_keypoint_overlay.mp4"

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    frame_idx = 0

    while cap.isOpened() and frame_idx < len(kp):
        ret, frame = cap.read()
        if not ret:
            break

        landmarks = kp[frame_idx]

        for start, end in HAND_CONNECTIONS:
            x1 = int(landmarks[start][0] * width)
            y1 = int(landmarks[start][1] * height)
            x2 = int(landmarks[end][0] * width)
            y2 = int(landmarks[end][1] * height)

            cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

        for point in landmarks:
            x = int(point[0] * width)
            y = int(point[1] * height)
            cv2.circle(frame, (x, y), 4, (0, 0, 255), -1)

        cv2.putText(
            frame,
            f"{CLASS_NAME.upper()} | Frame {frame_idx+1}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1,
            (255, 255, 255),
            2
        )

        out.write(frame)
        frame_idx += 1

    cap.release()
    out.release()

    print("Saved:", output_path)

In [ ]:
# CELL: DISPLAY HELP + DANGER VIDEOS

from IPython.display import Video, display

print("HELP Video:")
display(Video("help_keypoint_overlay.mp4", embed=True, width=500))

print("DANGER Video:")
display(Video("danger_keypoint_overlay.mp4", embed=True, width=500))

In [ ]:
# CELL: CREATE OUTPUT FOLDER

import os

OUTPUT_DIR = "/Users/venubandi/Desktop/Gesture_Project/outputs/keypoint_videos"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output folder ready:", OUTPUT_DIR)

In [ ]:
!pip install nbconvert

In [ ]:
!ls -R /Users/venubandi/Desktop/Gesture_Project | grep hand_keypoints_lstm_model

In [ ]:
!find /Users/venubandi/Desktop/Gesture_Project -name "*.ipynb"

In [ ]:
# CELL: DYNAMIC RESULTS (USING ACTUAL VARIABLES)

import os
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# ====== RECOMPUTE METRICS FROM MODEL ======
model.eval()

with torch.no_grad():
    outputs = model(X_test_tensor)
    probs = F.softmax(outputs, dim=1)
    _, preds = torch.max(outputs, 1)

# Accuracy
accuracy = accuracy_score(y_test_tensor.numpy(), preds.numpy())

# Classification report
report = classification_report(
    y_test_tensor.numpy(),
    preds.numpy(),
    target_names=["DANGER", "HELP"]
)

# Confusion matrix
conf_mat = confusion_matrix(
    y_test_tensor.numpy(),
    preds.numpy()
)

# ====== SAMPLE PREDICTION ======
sample = X_test_tensor[0].unsqueeze(0)

with torch.no_grad():
    out = model(sample)
    prob = F.softmax(out, dim=1)
    conf, pred = torch.max(prob, 1)

label_map = {0: "DANGER", 1: "HELP"}

actual_label = label_map[y_test_tensor[0].item()]
predicted_label = label_map[pred.item()]
confidence = conf.item() * 100

# ====== SPEED ======
import time

num_runs = 100
start = time.time()

with torch.no_grad():
    for _ in range(num_runs):
        _ = model(sample)

end = time.time()

inference_time = (end - start) / num_runs
fps = 1 / inference_time

# ====== BUILD TEXT ======
results_text = f"""
==============================
HAND KEYPOINTS LSTM RESULTS
==============================

Running Model:
LSTM gesture classifier using MediaPipe hand keypoints

Dataset:
Total samples: {len(X)}
Classes: DANGER, HELP

Weights:
Model trained and loaded successfully

Inference:
Keypoint sequence passed through trained model

Prediction Example:
Actual Label: {actual_label}
Predicted Label: {predicted_label}
Confidence: {confidence:.2f}%

Performance:
Test Accuracy: {accuracy * 100:.2f}%

Classification Report:
{report}

Confusion Matrix:
{conf_mat}

Speed:
Average inference time: {inference_time:.6f} seconds
Estimated FPS: {fps:.2f}

Metrics Used:
Accuracy, Precision, Recall, F1-score, Confusion Matrix, Inference Time, FPS

Notes:
- DANGER detection is strong
- HELP is sometimes confused with DANGER
- Model uses hand keypoints only (no body context)
"""

# ====== PRINT ======
print(results_text)

# ====== SAVE ======
output_path = "/Users/venubandi/Desktop/Gesture_Project/hand_keypoints_results.txt"

with open(output_path, "w") as f:
    f.write(results_text)

print("\nSaved to:", output_path)